# Police Stations Around College Park, MD
This notebook uses the OpenStreetMap Overpass API to find all police stations around College Park.


In [ ]:
# !pip install requests pandas folium
import requests
import pandas as pd
import folium


   -------------------- ------------------- 1/2 [folium]
   ---------------------------------------- 2/2 [folium]



In [3]:
CENTER_LAT = 38.9897
CENTER_LON = -76.9378
RADIUS_METERS = 5000

In [4]:
overpass_url = "https://overpass-api.de/api/interpreter"
query = f"""
[out:json][timeout:25];
(
  node["amenity"="police"](around:{RADIUS_METERS},{CENTER_LAT},{CENTER_LON});
  way["amenity"="police"](around:{RADIUS_METERS},{CENTER_LAT},{CENTER_LON});
  relation["amenity"="police"](around:{RADIUS_METERS},{CENTER_LAT},{CENTER_LON});
);
out center;
"""
response = requests.get(overpass_url, params={"data": query})
data = response.json()
len(data.get('elements', []))

11

In [5]:
elements = data.get('elements', [])
records = []
for el in elements:
    tags = el.get('tags', {})
    if 'lat' in el and 'lon' in el:
        lat, lon = el['lat'], el['lon']
    else:
        c = el.get('center', {})
        lat, lon = c.get('lat'), c.get('lon')
    records.append({
        'name': tags.get('name'),
        'lat': lat,
        'lon': lon,
        'operator': tags.get('operator')
    })
stations_df = pd.DataFrame(records)
stations_df

,name,lat,lon,operator
0,Hyattsville Police Station,38.952964,-76.941885,None
1,Berwyn Heights Police Station,38.993464,-76.921398,Berwyn Heights Town
2,Maryland National Capital Park Police Headquar...,38.962810,-76.902280,None
3,Town of Riverdale Park Police Station,38.961926,-76.928533,None
4,Prince George's County Police Department: Spec...,38.981908,-76.926522,Prince George's County Police Department
5,University Park Police,38.972330,-76.938372,None
6,University of Maryland Police Department,38.982958,-76.937146,University of Maryland College Park
7,Maryland State Police - Barrack Q,39.019654,-76.924341,Maryland State Police
8,Greenbelt Police Department,39.006602,-76.891439,None
9,Prince George's County Police Department Divis...,38.950820,-76.942795,Prince George's County Police Department


In [7]:
stations_df.to_csv("college_park_police_stations.csv", index=False)
print("Saved to college_park_police_stations.csv")

Saved to college_park_police_stations.csv


In [6]:
m = folium.Map(location=[CENTER_LAT, CENTER_LON], zoom_start=13)
for _, r in stations_df.iterrows():
    if pd.notna(r['lat']) and pd.notna(r['lon']):
        folium.Marker([r['lat'], r['lon']], tooltip=r['name']).add_to(m)
m